# Magnetometer Calibration Notebook

This notebook walks you through loading raw magnetometer data from the OpenLog Artemis, visualizing it, and computing hard iron calibration offsets.

**Before running:** make sure your `magcaldata.csv` file is in the `ola_data/` folder at the repo root.

## Step 1 — Load the data

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import sqrtm

path = '../ola_data/magcaldata.csv'

rows = []
with open(path, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append({k: float(v) if k not in ('Timestamp', 'Sensor') else v
                     for k, v in row.items()})

print(f'Loaded {len(rows)} samples')

mx = [r['MagX'] for r in rows]
my = [r['MagY'] for r in rows]
mz = [r['MagZ'] for r in rows]
t  = list(range(len(rows)))
data = np.column_stack([mx, my, mz])

In [ ]:
# magcal — magnetometer calibration (hard iron + soft iron)
# Port of the MathWorks MATLAB magcal function (Copyright 2018-2022 MathWorks).
#
# Usage:
#   A, b, magB = magcal(data)          # auto-select best fit
#   A, b, magB = magcal(data, 'eye')   # hard iron only
#   A, b, magB = magcal(data, 'diag')  # diagonal soft iron
#   A, b, magB = magcal(data, 'sym')   # full symmetric soft iron
#
#   corrected = (data - b) @ A

def _smallest_eigenvector(d):
    _, _, Vt = np.linalg.svd(d, full_matrices=False)
    return Vt[-1]

def _residual(Winv, V, B, data):
    spherept = (Winv @ (data.T - V.reshape(3, 1))).T
    return np.sum(spherept ** 2, axis=1) - B ** 2

def _is_pd(A):
    try:
        np.linalg.cholesky(A)
        return True
    except np.linalg.LinAlgError:
        return False

def _correct4(x, y, z):
    """4-parameter fit: A = I (hard iron offset only)."""
    bv = x**2 + y**2 + z**2
    A_mat = np.column_stack([x, y, z, np.ones(len(x))])
    soln, _, _, _ = np.linalg.lstsq(A_mat, bv, rcond=None)
    Winv = np.eye(3)
    V = 0.5 * soln[:3]
    B = np.sqrt(soln[3] + np.sum(V**2))
    res = A_mat @ soln - bv
    er = (1.0 / (2.0 * B * B)) * np.sqrt(res @ res / len(x))
    return Winv, V, B, er, True

def _correct7(x, y, z):
    """7-parameter fit: A = diagonal (diagonal soft iron)."""
    d = np.column_stack([x**2, y**2, z**2, x, y, z, np.ones(len(x))])
    beta = _smallest_eigenvector(d)
    A = np.diag(beta[:3])
    dA = beta[0] * beta[1] * beta[2]
    if dA < 0:
        A, beta, dA = -A, -beta, -dA
    V = -0.5 * (beta[3:6] / beta[:3])
    B = np.sqrt(abs(A[0,0]*V[0]**2 + A[1,1]*V[1]**2 + A[2,2]*V[2]**2 - beta[-1]))
    det3root = dA ** (1/3)
    Winv = sqrtm(A / det3root).real
    B /= np.sqrt(det3root)
    res = _residual(Winv, V, B, np.column_stack([x, y, z]))
    er = float(np.real((1.0 / (2*B*B)) * np.sqrt(res @ res / len(x))))
    return Winv, V, B, er, _is_pd(A)

def _correct10(x, y, z):
    """10-parameter fit: A = symmetric (full soft iron)."""
    d = np.column_stack([x*x, 2*x*y, 2*x*z, y*y, 2*y*z, z*z, x, y, z, np.ones(len(x))])
    beta = _smallest_eigenvector(d)
    A = np.array([[beta[0], beta[1], beta[2]],
                  [beta[1], beta[3], beta[4]],
                  [beta[2], beta[4], beta[5]]])
    dA = np.linalg.det(A)
    if dA < 0:
        A, beta, dA = -A, -beta, -dA
    V = -0.5 * np.linalg.solve(A, beta[6:9])
    B = np.sqrt(abs(A[0,0]*V[0]**2 + 2*A[1,0]*V[1]*V[0] + 2*A[2,0]*V[2]*V[0]
                  + A[1,1]*V[1]**2 + 2*A[2,1]*V[1]*V[2] + A[2,2]*V[2]**2 - beta[-1]))
    det3root = dA ** (1/3)
    Winv = sqrtm(A / det3root).real
    B /= np.sqrt(det3root)
    res = _residual(Winv, V, B, np.column_stack([x, y, z]))
    er = float(np.real((1.0 / (2*B*B)) * np.sqrt(res @ res / len(x))))
    return Winv, V, B, er, _is_pd(A)

def magcal(d, fitkind='auto'):
    d = np.asarray(d, dtype=float)
    if d.ndim != 2 or d.shape[1] != 3:
        raise ValueError("d must be an N×3 array")
    x, y, z = d[:,0], d[:,1], d[:,2]
    fitters = {'eye': _correct4, 'diag': _correct7, 'sym': _correct10}
    if fitkind in fitters:
        A, b, magB, _, _ = fitters[fitkind](x, y, z)
    elif fitkind == 'auto':
        A, b, magB, er, _ = _correct4(x, y, z)
        for fn in (_correct7, _correct10):
            Ai, bi, magBi, eri, ispd = fn(x, y, z)
            if ispd and np.all(np.isreal(Ai)) and eri < er:
                A, b, magB, er = Ai, bi, magBi, eri
    else:
        raise ValueError("fitkind must be 'eye', 'diag', 'sym', or 'auto'")
    return A, np.asarray(b).flatten(), magB

print('magcal ready')

## Step 2 — Plot the raw magnetometer data

A well-sampled, uncalibrated magnetometer traces an **off-center ellipsoid** in 3D space. The offset from the origin is caused by hard iron distortion (permanent magnetic fields near the sensor). Your goal is to find and remove that offset so the data forms a sphere centered at the origin.

In [ ]:
# 3D scatter — raw data
fig = plt.figure(figsize=(6, 5))
ax3d = fig.add_subplot(111, projection='3d')
ax3d.scatter(mx, my, mz, s=2, c=t, cmap='viridis')
ax3d.set_title('Raw Mag 3D')
ax3d.set_xlabel('MagX (µT)')
ax3d.set_ylabel('MagY (µT)')
ax3d.set_zlabel('MagZ (µT)')
plt.tight_layout()
plt.show()

In [ ]:
# XY and XZ projections — raw data
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(mx, my, s=2, c=t, cmap='viridis')
axes[0].set_aspect('equal')
axes[0].set_title('Raw Mag XY')
axes[0].set_xlabel('MagX (µT)'); axes[0].set_ylabel('MagY (µT)')
axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(0, color='k', lw=0.5)

axes[1].scatter(mx, mz, s=2, c=t, cmap='viridis')
axes[1].set_aspect('equal')
axes[1].set_title('Raw Mag XZ')
axes[1].set_xlabel('MagX (µT)'); axes[1].set_ylabel('MagZ (µT)')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()

## Step 3 — Compute hard iron offsets

The hard iron offset on each axis is the midpoint between the maximum and minimum values measured:

$$\text{offset}_x = \frac{\max(X) + \min(X)}{2}$$

Repeat for Y and Z. Fill in the cells below.

In [ ]:
# TODO: compute the min and max for each axis
mx_min = 
mx_max = 
my_min = 
my_max = 
mz_min = 
mz_max = 

print(f'MagX range: {mx_min:.2f} to {mx_max:.2f} µT')
print(f'MagY range: {my_min:.2f} to {my_max:.2f} µT')
print(f'MagZ range: {mz_min:.2f} to {mz_max:.2f} µT')

In [ ]:
# TODO: compute the hard iron offsets (midpoint of each axis range)
offset_x = 
offset_y = 
offset_z = 

print(f'Hard iron offsets: X={offset_x:.3f}, Y={offset_y:.3f}, Z={offset_z:.3f} µT')

## Step 4 — Apply the calibration

Subtract the offsets from the raw measurements to center the data at the origin.

In [ ]:
# TODO: apply the offsets to produce calibrated mag vectors
mx_cal = 
my_cal = 
mz_cal = 

## Step 5 — Plot the calibrated data

After calibration the 3D scatter should look like a sphere centered near the origin. If it still looks like an ellipsoid (stretched in one direction), you may need soft iron correction too.

In [ ]:
# 3D scatter — calibrated
fig = plt.figure(figsize=(6, 5))
ax3d = fig.add_subplot(111, projection='3d')
ax3d.scatter(mx_cal, my_cal, mz_cal, s=2, c=t, cmap='plasma')
ax3d.set_title('Calibrated Mag 3D')
ax3d.set_xlabel('MagX (µT)')
ax3d.set_ylabel('MagY (µT)')
ax3d.set_zlabel('MagZ (µT)')
plt.tight_layout()
plt.show()

In [ ]:
# XY and XZ projections — calibrated
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(mx_cal, my_cal, s=2, c=t, cmap='plasma')
axes[0].set_aspect('equal')
axes[0].set_title('Calibrated Mag XY')
axes[0].set_xlabel('MagX (µT)'); axes[0].set_ylabel('MagY (µT)')
axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(0, color='k', lw=0.5)

axes[1].scatter(mx_cal, mz_cal, s=2, c=t, cmap='plasma')
axes[1].set_aspect('equal')
axes[1].set_title('Calibrated Mag XZ')
axes[1].set_xlabel('MagX (µT)'); axes[1].set_ylabel('MagZ (µT)')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()

## Step 8 — Record your calibration values

Record both sets of results. You will use the `magcal` values in the sensor fusion notebook and firmware.

### Hard iron offsets

| Axis | Simple (midpoint) | magcal |
|------|-------------------|--------|
| X    |                   |        |
| Y    |                   |        |
| Z    |                   |        |

### Soft iron correction matrix (A)

```
[[ _______  _______  _______ ]
 [ _______  _______  _______ ]
 [ _______  _______  _______ ]]
```

Expected field strength: _______ µT

**Discussion questions:**
1. How close are your manual hard iron offsets to the `magcal` values?
2. Is the soft iron matrix `A` close to identity? What does that tell you about the sensor environment?
3. Does the `magcal` XY projection look rounder than the simple correction? Why or why not?
4. What happens to your heading estimate if you skip calibration entirely?

In [ ]:
A_mc, b_mc, magB = magcal(data)

print('Hard iron offsets from magcal (b):')
print(f'  offset_x = {b_mc[0]:.4f} µT')
print(f'  offset_y = {b_mc[1]:.4f} µT')
print(f'  offset_z = {b_mc[2]:.4f} µT')
print()
print(f'Your manual offsets:  X={offset_x:.4f},  Y={offset_y:.4f},  Z={offset_z:.4f} µT')
print()
print('Soft iron correction matrix (A):')
print(np.array2string(A_mc, precision=6, suppress_small=True))
print()
print(f'Expected field strength: {magB:.4f} µT')

In [ ]:
data_mc = (data - b_mc) @ A_mc
mx_mc = data_mc[:, 0].tolist()
my_mc = data_mc[:, 1].tolist()
mz_mc = data_mc[:, 2].tolist()

## Step 7 — Compare: simple vs full calibration

The plots below show your simple hard iron correction (left) next to the full `magcal` correction (right).

Look at the XY projection — if `magcal` produces a rounder circle, soft iron correction made a meaningful difference. If the two look the same, hard iron was the dominant distortion and the soft iron matrix is close to identity.

In [ ]:
fig = plt.figure(figsize=(14, 10))
fig.suptitle('Simple (hard iron only)  vs  magcal (hard + soft iron)', fontsize=12)

# Simple — 3D
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
ax1.scatter(mx_cal, my_cal, mz_cal, s=2, c=t, cmap='plasma')
ax1.set_title('Simple — 3D')
ax1.set_xlabel('MagX (µT)'); ax1.set_ylabel('MagY (µT)'); ax1.set_zlabel('MagZ (µT)')

# magcal — 3D
ax2 = fig.add_subplot(2, 3, 2, projection='3d')
ax2.scatter(mx_mc, my_mc, mz_mc, s=2, c=t, cmap='viridis')
ax2.set_title('magcal — 3D')
ax2.set_xlabel('MagX (µT)'); ax2.set_ylabel('MagY (µT)'); ax2.set_zlabel('MagZ (µT)')

# Simple — XY
ax3 = fig.add_subplot(2, 3, 4)
ax3.scatter(mx_cal, my_cal, s=2, c=t, cmap='plasma')
ax3.set_aspect('equal'); ax3.set_title('Simple — XY')
ax3.set_xlabel('MagX (µT)'); ax3.set_ylabel('MagY (µT)')
ax3.axhline(0, color='k', lw=0.5); ax3.axvline(0, color='k', lw=0.5)

# magcal — XY
ax4 = fig.add_subplot(2, 3, 5)
ax4.scatter(mx_mc, my_mc, s=2, c=t, cmap='viridis')
ax4.set_aspect('equal'); ax4.set_title('magcal — XY')
ax4.set_xlabel('MagX (µT)'); ax4.set_ylabel('MagY (µT)')
ax4.axhline(0, color='k', lw=0.5); ax4.axvline(0, color='k', lw=0.5)

# Simple — XZ
ax5 = fig.add_subplot(2, 3, 3)
ax5.scatter(mx_cal, mz_cal, s=2, c=t, cmap='plasma')
ax5.set_aspect('equal'); ax5.set_title('Simple — XZ')
ax5.set_xlabel('MagX (µT)'); ax5.set_ylabel('MagZ (µT)')
ax5.axhline(0, color='k', lw=0.5); ax5.axvline(0, color='k', lw=0.5)

# magcal — XZ
ax6 = fig.add_subplot(2, 3, 6)
ax6.scatter(mx_mc, mz_mc, s=2, c=t, cmap='viridis')
ax6.set_aspect('equal'); ax6.set_title('magcal — XZ')
ax6.set_xlabel('MagX (µT)'); ax6.set_ylabel('MagZ (µT)')
ax6.axhline(0, color='k', lw=0.5); ax6.axvline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()

## Step 6 — Record your calibration values

Write down your hard iron offsets — you will need them in the firmware and the sensor fusion notebook.

| Axis | Offset (µT) |
|------|-------------|
| X    |             |
| Y    |             |
| Z    |             |

**Discussion questions:**
1. Does your 3D scatter after calibration look like a sphere? If not, what shape is it and what might cause that?
2. How complete is your coverage of orientations? Are there gaps in the scatter plot?
3. What happens to your heading estimate if you skip calibration?